In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

colunas = [
    "timestamp",
    "solucao",
    "caso",
    "execucao",
    "filosofo",
    "grau",
    "bebeu",
    "tempo_total_execucao",
    "tempo_tranquilo",
    "tempo_com_sede",
    "tempo_bebendo",
    "espera_media_sede",
    "tentativas_falhas",
]

df = pd.read_csv(
    "resultados/resultados_randomized_backoff.txt",
    sep=";",
    header=None,
    names=colunas
)

df.head()

,2026-05-11 20:34:10,Randomized Backoff,caso1,1,1.1,2,6,18.126053899992257,5.838681200082647,5.123660599987488,6.0026645999460015,0.853943433331248,22
0,2026-05-11 20:34:10,Randomized Backoff,caso1,1,2,2,6,18.126054,5.796410,6.241883,6.001520,1.040314,31
1,2026-05-11 20:34:10,Randomized Backoff,caso1,1,3,2,6,18.126054,6.783731,3.691368,6.000670,0.615228,20
2,2026-05-11 20:34:10,Randomized Backoff,caso1,1,4,2,6,18.126054,4.723185,3.867491,6.001101,0.644582,17
3,2026-05-11 20:34:10,Randomized Backoff,caso1,1,5,2,6,18.126054,7.100989,5.021150,6.002698,0.836858,23
4,2026-05-11 20:34:29,Randomized Backoff,caso1,2,1,2,6,19.413042,8.850667,2.176446,6.001174,0.362741,11


In [5]:
df.groupby(["caso", "execucao", "filosofo"])["bebeu"].first().reset_index()

KeyError: 'caso'

In [ ]:
df_validacao = df.groupby(["caso", "execucao", "filosofo"])["bebeu"].first().reset_index()

df_validacao.groupby("caso")["bebeu"].describe()

In [ ]:
media_sede = (
    df.groupby(["caso", "filosofo"])["espera_media_sede"]
    .mean()
    .reset_index()
)

media_sede

In [ ]:
for caso in media_sede["caso"].unique():
    dados = media_sede[media_sede["caso"] == caso]

    plt.figure(figsize=(8, 4))
    plt.bar(dados["filosofo"], dados["espera_media_sede"])
    plt.xlabel("Filósofo")
    plt.ylabel("Espera média com sede (s)")
    plt.title(f"Espera média com sede por filósofo — {caso}")
    plt.xticks(dados["filosofo"])
    plt.grid(axis="y", alpha=0.3)
    plt.show()

In [ ]:
tempo_total = (
    df.groupby(["caso", "execucao"])["tempo_total_execucao"]
    .first()
    .reset_index()
)

tempo_total.head()

In [ ]:
for caso in tempo_total["caso"].unique():
    dados = tempo_total[tempo_total["caso"] == caso]

    plt.figure(figsize=(8, 4))
    plt.plot(dados["execucao"], dados["tempo_total_execucao"], marker="o")
    plt.xlabel("Execução")
    plt.ylabel("Tempo total (s)")
    plt.title(f"Tempo total por execução — {caso}")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
resumo_casos = (
    tempo_total.groupby("caso")["tempo_total_execucao"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

resumo_casos

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(resumo_casos["caso"], resumo_casos["mean"], yerr=resumo_casos["std"], capsize=5)
plt.xlabel("Caso")
plt.ylabel("Tempo total médio (s)")
plt.title("Comparação do tempo total médio entre casos")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
starvation = (
    df.groupby(["caso", "execucao"])["espera_media_sede"]
    .agg(["min", "max", "mean", "std"])
    .reset_index()
)

starvation["razao_max_min"] = starvation["max"] / starvation["min"].replace(0, pd.NA)

starvation.head()

In [ ]:
resumo_starvation = (
    starvation.groupby("caso")["razao_max_min"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

resumo_starvation

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(
    resumo_starvation["caso"],
    resumo_starvation["mean"],
    yerr=resumo_starvation["std"],
    capsize=5
)
plt.xlabel("Caso")
plt.ylabel("Razão maior espera / menor espera")
plt.title("Indicador de desequilíbrio de espera")
plt.grid(axis="y", alpha=0.3)
plt.show()